# 🤟 SANA A-PSL: Automated Video Keypoint Extraction & Data Synthesis Pipeline
**Goal:** Ingest raw video recordings (MP4, AVI, MOV, WEBM) of Pakistani Sign Language (PSL) / Medical phrases, extract high-precision **208-dimensional MediaPipe landmark sequences**, apply **6× Data Augmentation**, and format the dataset for SANA Conv1D Few-Shot fine-tuning.

### 📐 208-Dimension Feature Layout per Frame:
- `[0:66]`   : 33 Pose landmarks $(X, Y)$
- `[66:108]` : 21 Left Hand landmarks $(X, Y)$
- `[108:150]`: 21 Right Hand landmarks $(X, Y)$
- `[150:208]`: 29 Expression Face landmarks / Neutral 0s $(X, Y)$

### ⚙️ Pipeline Highlights:
1. **Universal Video Ingestion:** Handles webcam, mobile, or studio recordings.
2. **Selfie-Mirror Auto-Correction:** Corrects left/right hand flipping if recorded in selfie mode.
3. **60/100-Frame Spline Resampling:** Normalizes variable recording speeds to exact model token parity.
4. **Synthesis Engine:** Multiplies 400 raw videos into 2,400+ augmented training samples.
5. **Visual Verification:** Plots skeleton overlays to confirm accurate tracking before training.

In [ ]:
# ── Cell 1: Install & Import Dependencies ─────────────────────────────────────
!pip install -q mediapipe opencv-python numpy matplotlib tqdm pandas

import os
import sys
import glob
import json
import math
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import mediapipe as mp

print(f"OpenCV Version:   {cv2.__version__}")
print(f"MediaPipe Version: {mp.__version__}")
print(f"NumPy Version:     {np.__version__}")
print("✅ Environment ready for Keypoint Extraction!")

In [ ]:
# ── Cell 2: Global Configuration ─────────────────────────────────────────────
CONFIG = {
    # Input / Output Paths
    "RAW_VIDEOS_DIR":       "./raw_videos",            # Folder containing raw video files or subfolders
    "OUTPUT_KEYPOINTS_DIR": "./processed_psl_dataset", # Where .npy keypoints will be saved
    "OUTPUT_ZIP_NAME":      "SANA_PSL_Keypoints_Dataset.zip",
    
    # Landmark Extraction Parameters
    "INPUT_DIM":            208,                       # 66 Pose + 42 LH + 42 RH + 58 Face
    "TARGET_FRAMES":        60,                        # Resample all videos to 60 frames (Matches SANA Conv1D)
    "MIN_DETECTION_CONF":   0.5,
    "MIN_TRACKING_CONF":    0.5,
    "SMOOTHING_ALPHA":      0.75,                      # Exponential moving average filter weight
    "MIRROR_CORRECTION":    True,                      # Set True if videos were recorded with mirrored selfie cam
    
    # Data Augmentation
    "ENABLE_AUGMENTATION":  True,                      # Multiply dataset with spatial & temporal synthesis
    "AUGMENTATION_FACTOR":  5,                         # 1 original + 5 augmented = 6x dataset expansion
    "SCALE_RANGE":          (0.85, 1.15),              # Random skeletal zoom (simulates distance from camera)
    "SHIFT_RANGE":          (-0.06, 0.06),             # Random translation (simulates signer position)
    "SPEED_RANGE":          (0.85, 1.15),              # Random temporal stretch/compression
    "JITTER_SIGMA":         0.003,                     # Gaussian coordinate jitter
}

# Ensure directories exist
os.makedirs(CONFIG["RAW_VIDEOS_DIR"], exist_ok=True)
os.makedirs(CONFIG["OUTPUT_KEYPOINTS_DIR"], exist_ok=True)

print("⚙️ Keypoints Pipeline Configuration Initialized:")
print(json.dumps(CONFIG, indent=2))

In [ ]:
# ── Cell 3: MediaPipe Holistic Landmark Extractor ────────────────────────────
class MediaPipeVideoExtractor:
    def __init__(self, min_detection_conf=0.5, min_tracking_conf=0.5, mirror_fix=True, alpha=0.75):
        self.mp_holistic = mp.solutions.holistic
        self.holistic = self.mp_holistic.Holistic(
            static_image_mode=False,
            model_complexity=2,
            enable_segmentation=False,
            refine_face_landmarks=False,
            min_detection_confidence=min_detection_conf,
            min_tracking_confidence=min_tracking_conf
        )
        self.mirror_fix = mirror_fix
        self.alpha = alpha
        self.prev_landmarks = None
        
    def reset_tracker(self):
        self.prev_landmarks = None
        
    def extract_frame(self, frame_bgr):
        """
        Extracts exactly 208 normalized (x, y) coordinates from a single frame.
        Returns: np.ndarray shape (208,)
        """
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results = self.holistic.process(frame_rgb)
        
        # 1. Pose Landmarks (33 points -> 66 floats: x, y)
        pose_coords = [0.0] * 66
        if results.pose_landmarks:
            for i, lm in enumerate(results.pose_landmarks.landmark):
                pose_coords[i*2] = float(lm.x)
                pose_coords[i*2 + 1] = float(lm.y)
                
        # 2. Left Hand Landmarks (21 points -> 42 floats: x, y)
        lh_coords = [0.0] * 42
        if results.left_hand_landmarks:
            for i, lm in enumerate(results.left_hand_landmarks.landmark):
                lh_coords[i*2] = float(lm.x)
                lh_coords[i*2 + 1] = float(lm.y)
                
        # 3. Right Hand Landmarks (21 points -> 42 floats: x, y)
        rh_coords = [0.0] * 42
        if results.right_hand_landmarks:
            for i, lm in enumerate(results.right_hand_landmarks.landmark):
                rh_coords[i*2] = float(lm.x)
                rh_coords[i*2 + 1] = float(lm.y)
                
        # Handle Selfie Mirror Inversion if enabled
        if self.mirror_fix:
            lh_coords, rh_coords = rh_coords, lh_coords
            
        # 4. Face Landmarks (58 floats - default neutral zeros for SANA compatibility)
        face_coords = [0.0] * 58
        
        # Concatenate into 208 floats
        current_frame_208 = np.array(pose_coords + lh_coords + rh_coords + face_coords, dtype=np.float32)
        
        # Coordinate smoothing across consecutive frames
        if self.prev_landmarks is None:
            self.prev_landmarks = current_frame_208
        else:
            active_mask = (current_frame_208 != 0.0).astype(np.float32)
            smoothed = active_mask * (self.alpha * current_frame_208 + (1 - self.alpha) * self.prev_landmarks) + (1 - active_mask) * current_frame_208
            self.prev_landmarks = smoothed
            current_frame_208 = smoothed
            
        return current_frame_208

    def extract_from_video(self, video_path):
        """
        Ingests a video file and extracts the full temporal sequence of landmarks.
        Returns: np.ndarray of shape (T, 208), fps, total_frames
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"⚠️ Error: Could not open video {video_path}")
            return None, 0, 0
        
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        self.reset_tracker()
        sequence = []
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            landmarks_208 = self.extract_frame(frame)
            sequence.append(landmarks_208)
            
        cap.release()
        
        if len(sequence) == 0:
            return None, fps, 0
            
        return np.array(sequence, dtype=np.float32), fps, len(sequence)

    def close(self):
        self.holistic.close()

print("✅ MediaPipeVideoExtractor class defined successfully!")

In [ ]:
# ── Cell 4: Temporal Spline Resampling & Normalization ───────────────────────
def resample_sequence(sequence, target_frames=60):
    """
    Resamples an arbitrary length landmark sequence (T, 208) to exactly target_frames (e.g. 60)
    using linear/spline temporal interpolation.
    """
    T = sequence.shape[0]
    if T == target_frames:
        return sequence
    
    orig_times = np.linspace(0, 1, T)
    target_times = np.linspace(0, 1, target_frames)
    
    resampled = np.zeros((target_frames, sequence.shape[1]), dtype=np.float32)
    for dim in range(sequence.shape[1]):
        resampled[:, dim] = np.interp(target_times, orig_times, sequence[:, dim])
        
    return resampled

def normalize_skeleton(sequence_208):
    """
    Normalizes skeleton keypoints centered around the mid-shoulder point
    and scales by shoulder width for invariant signer body proportions.
    """
    # Pose 11: Left Shoulder (coords 22, 23), Pose 12: Right Shoulder (coords 24, 25)
    norm_seq = sequence_208.copy()
    for t in range(norm_seq.shape[0]):
        ls_x, ls_y = norm_seq[t, 22], norm_seq[t, 23]
        rs_x, rs_y = norm_seq[t, 24], norm_seq[t, 25]
        
        if ls_x != 0 and rs_x != 0:
            center_x = (ls_x + rs_x) / 2.0
            center_y = (ls_y + rs_y) / 2.0
            shoulder_dist = math.sqrt((ls_x - rs_x)**2 + (ls_y - rs_y)**2)
            if shoulder_dist > 0.01:
                # Scale hands and pose relative to center
                for i in range(0, 150, 2):
                    if norm_seq[t, i] != 0.0 or norm_seq[t, i+1] != 0.0:
                        norm_seq[t, i] = (norm_seq[t, i] - center_x) / shoulder_dist
                        norm_seq[t, i+1] = (norm_seq[t, i+1] - center_y) / shoulder_dist
                        
    return norm_seq

print("✅ Temporal Resampling & Normalization modules ready!")

In [ ]:
# ── Cell 5: 6× Data Synthesis & Augmentation Engine ──────────────────────────
class LandmarkAugmentor:
    def __init__(self, scale_range=(0.85, 1.15), shift_range=(-0.05, 0.05), speed_range=(0.85, 1.15), jitter_sigma=0.003):
        self.scale_range = scale_range
        self.shift_range = shift_range
        self.speed_range = speed_range
        self.jitter_sigma = jitter_sigma
        
    def augment(self, sequence_60):
        """
        Applies random spatial scaling, 2D translation, temporal speed warping, and jitter.
        Input:  (60, 208)
        Output: (60, 208) newly synthesized variation
        """
        aug_seq = sequence_60.copy()
        T = aug_seq.shape[0]
        
        # 1. Temporal Speed Warping (Stretch / Compress speed)
        speed_factor = random.uniform(*self.speed_range)
        warped_len = max(10, int(T * speed_factor))
        orig_t = np.linspace(0, 1, T)
        warp_t = np.linspace(0, 1, warped_len)
        warped = np.zeros((warped_len, 208), dtype=np.float32)
        for d in range(208):
            warped[:, d] = np.interp(warp_t, orig_t, aug_seq[:, d])
        aug_seq = resample_sequence(warped, target_frames=T)
        
        # 2. Spatial Scaling (Distance from camera)
        scale = random.uniform(*self.scale_range)
        
        # 3. Spatial Translation (Position shift)
        shift_x = random.uniform(*self.shift_range)
        shift_y = random.uniform(*self.shift_range)
        
        # 4. Coordinate Jitter
        jitter = np.random.normal(0, self.jitter_sigma, size=aug_seq.shape).astype(np.float32)
        
        for t in range(T):
            for i in range(0, 150, 2): # Apply to Pose, Left Hand, Right Hand
                if aug_seq[t, i] != 0.0 or aug_seq[t, i+1] != 0.0:
                    # Scale around center (0.5, 0.5)
                    x_centered = (aug_seq[t, i] - 0.5) * scale + 0.5 + shift_x + jitter[t, i]
                    y_centered = (aug_seq[t, i+1] - 0.5) * scale + 0.5 + shift_y + jitter[t, i+1]
                    aug_seq[t, i] = np.clip(x_centered, 0.0, 1.0)
                    aug_seq[t, i+1] = np.clip(y_centered, 0.0, 1.0)
                    
        return aug_seq

augmentor = LandmarkAugmentor(
    scale_range=CONFIG["SCALE_RANGE"],
    shift_range=CONFIG["SHIFT_RANGE"],
    speed_range=CONFIG["SPEED_RANGE"],
    jitter_sigma=CONFIG["JITTER_SIGMA"]
)
print("✅ Data Synthesis & Augmentation Engine ready!")

In [ ]:
# ── Cell 6: Main Video Extraction & Synthesis Pipeline ───────────────────────
def scan_video_files(root_dir):
    """
    Finds all video files and auto-detects class names from subfolder or filename prefix.
    """
    extensions = ('*.mp4', '*.avi', '*.mov', '*.webm', '*.mkv')
    video_files = []
    for ext in extensions:
        video_files.extend(glob.glob(os.path.join(root_dir, '**', ext), recursive=True))
        
    parsed_samples = []
    for vpath in sorted(video_files):
        rel_path = os.path.relpath(vpath, root_dir)
        parts = Path(rel_path).parts
        
        if len(parts) > 1:
            # Structure: raw_videos/<phrase_class>/video.mp4
            class_name = parts[0]
        else:
            # Structure: raw_videos/<phrase_class>_rep_01.mp4
            stem = Path(vpath).stem
            class_name = stem.split('_')[0] if '_' in stem else stem
            
        parsed_samples.append({
            "video_path": vpath,
            "class_name": class_name,
            "file_name": os.path.basename(vpath)
        })
        
    return parsed_samples

# Run Video Scanner
samples = scan_video_files(CONFIG["RAW_VIDEOS_DIR"])
print(f"🔍 Found {len(samples)} video files in '{CONFIG['RAW_VIDEOS_DIR']}'.")

if len(samples) == 0:
    print("\n💡 NOTE: No videos found in './raw_videos' yet.")
    print("👉 Place your recorded .mp4 files into './raw_videos' and re-run this cell!")
    print("Example structure:")
    print("  raw_videos/")
    print("    ├── headache/")
    print("    │   ├── rep_01.mp4")
    print("    │   └── rep_02.mp4")
    print("    └── fever/")
    print("        ├── rep_01.mp4")
    print("        └── rep_02.mp4")
else:
    extractor = MediaPipeVideoExtractor(
        min_detection_conf=CONFIG["MIN_DETECTION_CONF"],
        min_tracking_conf=CONFIG["MIN_TRACKING_CONF"],
        mirror_fix=CONFIG["MIRROR_CORRECTION"],
        alpha=CONFIG["SMOOTHING_ALPHA"]
    )
    
    dataset_records = []
    print("\n🚀 Starting Landmark Extraction & Augmentation Pipeline...")
    
    for item in tqdm(samples, desc="Processing Videos"):
        vpath = item["video_path"]
        cname = item["class_name"]
        fname = Path(item["file_name"]).stem
        
        # Extract Raw Landmarks
        raw_seq, fps, frame_count = extractor.extract_from_video(vpath)
        if raw_seq is None or frame_count < 5:
            print(f"⚠️ Skipping {vpath} (insufficient frames)")
            continue
            
        # Resample to exactly TARGET_FRAMES (60 frames)
        resampled_seq = resample_sequence(raw_seq, target_frames=CONFIG["TARGET_FRAMES"])
        
        # Output Directory per class
        class_out_dir = os.path.join(CONFIG["OUTPUT_KEYPOINTS_DIR"], cname)
        os.makedirs(class_out_dir, exist_ok=True)
        
        # 1. Save Original Clean Keypoints
        orig_out_path = os.path.join(class_out_dir, f"{fname}_orig.npy")
        np.save(orig_out_path, resampled_seq)
        dataset_records.append({
            "npy_path": orig_out_path,
            "class_name": cname,
            "type": "original",
            "raw_frames": frame_count,
            "resampled_frames": CONFIG["TARGET_FRAMES"],
            "fps": fps
        })
        
        # 2. Synthesize Augmented Variations (6x Total Expansion)
        if CONFIG["ENABLE_AUGMENTATION"]:
            for aug_idx in range(1, CONFIG["AUGMENTATION_FACTOR"] + 1):
                aug_seq = augmentor.augment(resampled_seq)
                aug_out_path = os.path.join(class_out_dir, f"{fname}_aug_{aug_idx:02d}.npy")
                np.save(aug_out_path, aug_seq)
                dataset_records.append({
                    "npy_path": aug_out_path,
                    "class_name": cname,
                    "type": f"augmented_{aug_idx}",
                    "raw_frames": frame_count,
                    "resampled_frames": CONFIG["TARGET_FRAMES"],
                    "fps": fps
                })
                
    extractor.close()
    print(f"\n🎉 Extraction Complete! Generated {len(dataset_records)} processed .npy files.")

In [ ]:
# ── Cell 7: Generate Metadata & Translation Mapping ──────────────────────────
# Standard Bilingual Medical Translation Mapping (English + Urdu)
MEDICAL_TRANSLATIONS = {
    "headache":         ("I have a severe headache", "میرے سر میں شدید درد ہے"),
    "stomach_pain":     ("My stomach hurts", "میرے پیٹ میں درد ہے"),
    "chest_pain":       ("I have chest pain", "میرے سینے میں درد ہے"),
    "fever":            ("I have a high fever", "مجھے تیز بخار ہے"),
    "cough":            ("I have a cough", "مجھے کھانسی ہے"),
    "dizziness":        ("I feel dizzy", "مجھے چکر آ رہے ہیں"),
    "nausea":           ("I feel nauseous", "مجھے متلی ہو رہی ہے"),
    "allergy":          ("I have an allergy", "مجھے الرجی ہے"),
    "diabetes":         ("I have diabetes", "مجھے شوگر ہے"),
    "blood_pressure":   ("My blood pressure is high", "میرا بلڈ پریشر زیادہ ہے"),
    "breathless":       ("I cannot breathe properly", "مجھے سانس لینے میں دشواری ہو رہی ہے"),
    "bleeding":         ("There is bleeding", "خون بہہ رہا ہے"),
    "broken_bone":      ("My bone is fractured", "میری ہڈی ٹوٹ گئی ہے"),
    "burn":             ("I have a burn injury", "میرا ہاتھ جل گیا ہے"),
    "medicine":         ("I need my medicine", "مجھے میری دوائی چاہیے"),
    "water":            ("Please give me water", "براہ کرم مجھے پانی دیں"),
    "doctor":           ("I need to see a doctor", "مجھے ڈاکٹر سے ملنا ہے"),
    "help":             ("Please help me", "براہ کرم میری مدد کریں"),
    "emergency":        ("This is an emergency", "یہ ایمرجنسی ہے"),
    "pain_scale_high":  ("The pain is very intense (10/10)", "درد بہت شدید ہے"),
}

metadata_csv_path = os.path.join(CONFIG["OUTPUT_KEYPOINTS_DIR"], "dataset_metadata.csv")

if 'dataset_records' in locals() and len(dataset_records) > 0:
    df = pd.DataFrame(dataset_records)
    
    # Add translations
    df["english_translation"] = df["class_name"].apply(lambda c: MEDICAL_TRANSLATIONS.get(c, (c.replace('_', ' ').title(), ""))[0])
    df["urdu_translation"]    = df["class_name"].apply(lambda c: MEDICAL_TRANSLATIONS.get(c, ("", ""))[1])
    
    df.to_csv(metadata_csv_path, index=False)
    print(f"📄 Dataset metadata saved to: {metadata_csv_path}")
    print("\n--- Class Sample Counts ---")
    print(df["class_name"].value_counts().to_string())
    display(df.head(10))
else:
    print("ℹ️ Note: Metadata will generate once raw video files are processed.")

In [ ]:
# ── Cell 8: Visual Skeleton Verification ─────────────────────────────────────
def plot_extracted_skeleton(npy_path, frame_indices=[0, 15, 30, 45, 59]):
    """
    Visualizes 2D skeletal pose and hands across multiple frames to verify tracking quality.
    """
    data = np.load(npy_path) # Shape: (60, 208)
    T = data.shape[0]
    
    fig, axes = plt.subplots(1, len(frame_indices), figsize=(18, 4))
    fig.suptitle(f"SANA Skeleton Tracking: {os.path.basename(npy_path)}", fontsize=14, y=1.05)
    
    for idx, f_idx in enumerate(frame_indices):
        f_idx = min(f_idx, T - 1)
        frame_data = data[f_idx]
        
        pose_xy = frame_data[0:66].reshape(-1, 2)
        lh_xy   = frame_data[66:108].reshape(-1, 2)
        rh_xy   = frame_data[108:150].reshape(-1, 2)
        
        ax = axes[idx]
        ax.set_title(f"Frame {f_idx}/{T}")
        ax.set_xlim(0, 1)
        ax.set_ylim(1, 0) # Invert Y for image coordinate standard
        ax.set_aspect('equal')
        ax.grid(True, linestyle='--', alpha=0.3)
        
        # Plot Pose (Upper Body)
        valid_pose = pose_xy[pose_xy.sum(axis=1) != 0]
        if len(valid_pose) > 0:
            ax.scatter(valid_pose[:, 0], valid_pose[:, 1], c='blue', s=25, label='Pose')
            
        # Plot Left Hand
        valid_lh = lh_xy[lh_xy.sum(axis=1) != 0]
        if len(valid_lh) > 0:
            ax.scatter(valid_lh[:, 0], valid_lh[:, 1], c='green', s=35, label='Left Hand')
            
        # Plot Right Hand
        valid_rh = rh_xy[rh_xy.sum(axis=1) != 0]
        if len(valid_rh) > 0:
            ax.scatter(valid_rh[:, 0], valid_rh[:, 1], c='red', s=35, label='Right Hand')
            
        if idx == 0:
            ax.legend(loc='lower right', fontsize=8)
            
    plt.tight_layout()
    plt.show()

# Test visualization on the first available sample
sample_files = glob.glob(os.path.join(CONFIG["OUTPUT_KEYPOINTS_DIR"], '**', '*.npy'), recursive=True)
if sample_files:
    print(f"Visualizing sample: {sample_files[0]}")
    plot_extracted_skeleton(sample_files[0])
else:
    print("ℹ️ Once videos are processed, running this cell displays skeleton verification strips.")

In [ ]:
# ── Cell 9: Package Dataset into ZIP for Kaggle / Cloud Training ─────────────
zip_output_path = os.path.join("./", CONFIG["OUTPUT_ZIP_NAME"].replace('.zip', ''))
dataset_folder = CONFIG["OUTPUT_KEYPOINTS_DIR"]

if os.path.exists(dataset_folder) and len(os.listdir(dataset_folder)) > 0:
    print(f"📦 Compressing '{dataset_folder}' into '{CONFIG['OUTPUT_ZIP_NAME']}'...")
    shutil.make_archive(zip_output_path, 'zip', dataset_folder)
    zip_size_mb = os.path.getsize(zip_output_path + ".zip") / (1024 * 1024)
    print(f"\n✅ Dataset Successfully Packaged!")
    print(f"📁 Archive: {zip_output_path}.zip ({zip_size_mb:.2f} MB)")
    print("🚀 Upload this ZIP directly to Kaggle / Google Colab to run Few-Shot fine-tuning!")
else:
    print("ℹ️ Run Cell 6 first to generate processed keypoint files before zipping.")